# CoT Synthesis Pipeline

Bikin data training dari soal mentah pakai **teacher distillation**:

```
merged_dataset.jsonl  (soal, cara, jawaban)
        |
  1. generate   -> teacher (Groq R1-distill-70b) bikin N solusi CoT per soal
        |          data/cot/candidates.jsonl
  2. filter     -> LLM judge (Groq llama-8b) buang solusi yang salah
        |          data/cot/correct.jsonl
  3. to_chatml  -> 2 dataset: cot.jsonl (reasoning) + nocot.jsonl (jawaban saja)
                   data/sft/
```

Tiap stage = config cell -> run cell -> inspect cell. Ubah `LIMIT` kecil dulu buat coba.

Butuh: `GROQ_API_KEY` (gratis di https://console.groq.com). Jalan di lokal / Colab / Kaggle.

## 0. Setup

In [1]:
# Colab / Kaggle: hilangkan komentar baris bawah. Lokal: skip kalau requirements.txt sudah terpasang.
# !pip install -q openai python-dotenv math-verify

In [2]:
import sys
from pathlib import Path

# Cari root repo (folder yang punya src/cot_synthesis) lalu daftarkan ke path import.
def find_repo_root(start: Path) -> Path:
    for cand in [start.resolve(), *start.resolve().parents]:
        if (cand / 'src' / 'cot_synthesis').exists():
            return cand
    raise RuntimeError('root repo (punya src/cot_synthesis) tidak ketemu')

REPO_ROOT = find_repo_root(Path.cwd())
sys.path.insert(0, str(REPO_ROOT))
print('repo root:', REPO_ROOT)

repo root: D:\Main Storage\Vscode\FP_NLP\FP_NLP


## 1. API key (Groq)

Taruh `GROQ_API_KEY=gsk_xxx` di file `.env` pada root repo, atau ketik pas diminta.

In [3]:
import os
try:
    from dotenv import load_dotenv
    load_dotenv(REPO_ROOT / '.env')
except Exception:
    pass

if not os.environ.get('GROQ_API_KEY'):
    from getpass import getpass
    os.environ['GROQ_API_KEY'] = getpass('GROQ_API_KEY: ')

print('GROQ key terpasang:', bool(os.environ.get('GROQ_API_KEY')))

GROQ key terpasang: True


## 2. Config

Knob utama. Buat coba: `LIMIT` kecil + `N_CANDIDATES` kecil biar hemat kuota. Produksi: `LIMIT=None`, `N_CANDIDATES=8`.

In [4]:
INPUT      = REPO_ROOT / 'data' / 'merged_dataset.jsonl'   # ganti ke file lain kalau perlu
COT_DIR    = REPO_ROOT / 'data' / 'cot'
SFT_DIR    = REPO_ROOT / 'data' / 'sft'
CANDIDATES = COT_DIR / 'candidates.jsonl'
CORRECT    = COT_DIR / 'correct.jsonl'

TEACHER_MODEL = 'llama-3.3-70b-versatile'  # teacher (Groq, robust free-tier). Alt: qwen/qwen3-32b (reasoning, TPM lebih kecil)
JUDGE_MODEL   = 'llama-3.1-8b-instant'     # judge (Groq, kecil+cepat)

N_CANDIDATES = 2      # solusi per soal (8 utk produksi)
LIMIT        = 5      # jumlah soal diproses; None = semua
SLEEP        = 2.0    # jeda detik antar call (Groq free-tier ~30 req/menit)
MAX_TOKENS   = 2048   # cukup utk solusi langkah + \boxed; aman di bawah limit TPM free-tier

assert INPUT.exists(), f'tidak ada: {INPUT}'
print('input :', INPUT)
print('limit :', LIMIT, '| N:', N_CANDIDATES, '| teacher:', TEACHER_MODEL)

input : D:\Main Storage\Vscode\FP_NLP\FP_NLP\data\merged_dataset.jsonl
limit : 5 | N: 2 | teacher: llama-3.3-70b-versatile


## 3. Generate teacher CoT

Resumable: kalau putus/kena rate-limit, jalankan ulang cell ini — yang sudah jadi dilewati.

In [5]:
from src.cot_synthesis.generate import run_generate

gen_stats = run_generate(
    INPUT, CANDIDATES, backend='api', model=TEACHER_MODEL,
    n=N_CANDIDATES, max_tokens=MAX_TOKENS, limit=LIMIT, sleep=SLEEP,
)
print(gen_stats)

{'problems': 25190, 'generated': 10, 'todo_problems': 5}


In [6]:
# Intip 1 kandidat: soal, jawaban gold, boxed hasil teacher, potongan reasoning.
from src.cot_synthesis.utils import read_jsonl, extract_boxed

cands = read_jsonl(CANDIDATES)
print('total kandidat:', len(cands))
r = cands[0]
print('soal :', r['soal'][:200])
print('gold :', r['jawaban'])
print('boxed:', extract_boxed(r['text']))
print('---- teacher text (800 char) ----')
print(r['text'][:800])

total kandidat: 10
soal : Tentukan translasi dari garis k dengan persamaan $y = x^2 - 2x - 8$ oleh $\begin{pmatrix} 0 \\ 4 \end{pmatrix}$.
gold : Hasil translasi dari garis k adalah $y = x^2 - 2x - 4$.
boxed: y = x^2 - 2x - 4
---- teacher text (800 char) ----
Untuk menentukan translasi dari garis k dengan persamaan $y = x^2 - 2x - 8$ oleh $\begin{pmatrix} 0 \\ 4 \end{pmatrix}$, kita perlu menerapkan vektor translasi ke persamaan garis tersebut.

Langkah 1: Tulis kembali persamaan garis dalam bentuk umum $y = f(x)$.
Persamaan garis k adalah $y = x^2 - 2x - 8$.

Langkah 2: Terapkan vektor translasi $\begin{pmatrix} 0 \\ 4 \end{pmatrix}$ ke persamaan garis.
Vektor translasi hanya memiliki komponen y, sehingga hanya akan memengaruhi nilai y. Kita dapat menerapkan translasi dengan menambahkan 4 ke nilai y.

Langkah 3: Tulis persamaan baru setelah translasi.
$y' = x^2 - 2x - 8 + 4$
$y' = x^2 - 2x - 4$

Jadi, persamaan garis setelah translasi adalah $y = x^2 - 2x - 4$.

Jawaban akhir adalah $

## 4. Filter (LLM judge)

Buang kandidat tanpa `\boxed`, lalu judge bandingin prediksi vs `jawaban`. Simpan yang benar saja.

In [7]:
from src.cot_synthesis.filter_solutions import run_filter

flt_stats = run_filter(CANDIDATES, CORRECT, judge_model=JUDGE_MODEL, sleep=1.0)
acc = flt_stats['kept'] / max(flt_stats['total'], 1)
print(flt_stats)
print(f'acceptance rate: {acc:.1%}  (cek apakah N kandidat cukup)')

{'total': 10, 'no_boxed': 0, 'no_gold': 0, 'wrong': 2, 'kept': 8, 'by_prefilter': 0, 'by_judge': 8, 'problems_covered': 4}
acceptance rate: 80.0%  (cek apakah N kandidat cukup)


In [8]:
correct = read_jsonl(CORRECT)
print('solusi benar:', len(correct), '| soal tercakup:', len({c['id'] for c in correct}))
for c in correct[:2]:
    print('soal :', c['soal'][:120])
    print('pred :', c['pred'], '| gold:', c['jawaban'])
    print()

solusi benar: 8 | soal tercakup: 4
soal : Tentukan translasi dari garis k dengan persamaan $y = x^2 - 2x - 8$ oleh $\begin{pmatrix} 0 \\ 4 \end{pmatrix}$.
pred : y = x^2 - 2x - 4 | gold: Hasil translasi dari garis k adalah $y = x^2 - 2x - 4$.

soal : Tentukan translasi dari garis k dengan persamaan $y = x^2 - 2x - 8$ oleh $\begin{pmatrix} 0 \\ 4 \end{pmatrix}$.
pred : y = x^2 - 2x - 4 | gold: Hasil translasi dari garis k adalah $y = x^2 - 2x - 4$.



## 5. Build ChatML (cot + nocot)

Dari solusi benar yang SAMA bikin 2 dataset: `cot.jsonl` (penuh reasoning) dan `nocot.jsonl` (jawaban saja). Ini inti eksperimen CoT vs non-CoT.

In [9]:
from src.cot_synthesis.to_chatml import run as build_chatml

ml_stats = build_chatml(CORRECT, SFT_DIR, max_solution_chars=8000)
print(ml_stats)

{'input_rows': 8, 'cot_examples': 8, 'nocot_examples': 8, 'problems': 4, 'skipped_long': 0}


In [10]:
# Banding 1 soal: target CoT vs target non-CoT.
cot   = read_jsonl(SFT_DIR / 'cot.jsonl')
nocot = read_jsonl(SFT_DIR / 'nocot.jsonl')
print('cot:', len(cot), '| nocot:', len(nocot), '\n')

m = cot[0]['messages']
print('=== COT ===')
print('USER     :', m[0]['content'][:200])
print('ASSISTANT:', m[1]['content'][:500])

m2 = nocot[0]['messages']
print('\n=== NON-COT (soal sama) ===')
print('USER     :', m2[0]['content'][:200])
print('ASSISTANT:', m2[1]['content'])

cot: 8 | nocot: 8 

=== COT ===
USER     : Selesaikan soal matematika berikut. Tunjukkan langkah-langkah penyelesaian secara rinci dan sistematis dalam Bahasa Indonesia. Pastikan jawaban akhir (dan hanya jawaban) berada di dalam \boxed{}.

Ten
ASSISTANT: Untuk menentukan translasi dari garis k dengan persamaan $y = x^2 - 2x - 8$ oleh $\begin{pmatrix} 0 \\ 4 \end{pmatrix}$, kita perlu menerapkan vektor translasi ke persamaan garis tersebut.

Langkah 1: Tulis kembali persamaan garis dalam bentuk umum $y = f(x)$.
Persamaan garis k adalah $y = x^2 - 2x - 8$.

Langkah 2: Terapkan vektor translasi $\begin{pmatrix} 0 \\ 4 \end{pmatrix}$ ke persamaan garis.
Vektor translasi hanya memiliki komponen y, sehingga hanya akan memengaruhi nilai y. Kita dapat m

=== NON-COT (soal sama) ===
USER     : Selesaikan soal matematika berikut. Berikan HANYA jawaban akhir di dalam \boxed{} tanpa langkah-langkah.

Tentukan translasi dari garis k dengan persamaan $y = x^2 - 2x - 8$ oleh $\begin{pmatrix} 0 \\
ASS

## 6. Selesai

Output: `data/sft/cot.jsonl` + `data/sft/nocot.jsonl` siap buat SFT.

Lanjut: naikin `LIMIT=None` + `N_CANDIDATES=8` buat full run, lalu training (`src/training/`).